In [ ]:
!pip install rapidfuzz --quiet

In [ ]:
import pandas as pd

url = "https://data.opensanctions.org/artifacts/us_ofac_sdn/20261024141004-hou/targets.simple.csv"
sanctions_df = pd.read_csv(url)

print(f"Завантажено записів: {len(sanctions_df)}")
sanctions_df.head()

Завантажено записів: 20434


,id,schema,name,aliases,birth_date,countries,addresses,identifiers,sanctions,phones,emails,program_ids,dataset,first_seen,last_seen,last_change
0,NK-223CQDBzp8MRkdJMDiqXn3,Organization,"Myanmar Yatai International Holding Group Co.,...",Shwe Kokko Special Economic Zone;Yatai New Cit...,NaN,mm,"Hpa-An City;Shwe Kokko Village, Myawaddy Towns...",103919088,GLOMAG - Executive Order 13818 (Global Magnitsky),NaN,NaN,US-GLOMAG,US OFAC Specially Designated Nationals (SDN) List,2025-09-08T14:10:01,2026-10-24T14:10:04,2026-01-26T16:10:01
1,NK-22HtK7WrxZ2sU3rmhz6PuZ,Person,Michael Kuajien,Michael Kuajian;Michael Kuajien Duer Mayok,1979-01-01,ke;ss,Nairobi,NaN,GLOMAG - Executive Order 13818 (Global Magnitsky),NaN,NaN,US-GLOMAG,US OFAC Specially Designated Nationals (SDN) List,2023-04-20T10:27:20,2026-10-24T14:10:04,2026-01-26T16:10:01
2,NK-22MHXvQQufBgTjUWgUbWb8,Organization,Limited Liability Company Specialized Develope...,LLC Specialized Developer Alabuga South Park;O...,NaN,ru,"ul. Sh-2 (OEZ Alabuga Ter.), D. 15/5, Pomeshch...",1231600029825;1674005078;52120883,RUSSIA-EO14024 - Executive Order 14024 (Russia),NaN,NaN,US-RUSHAR,US OFAC Specially Designated Nationals (SDN) List,2024-02-23T14:10:55,2026-10-24T14:10:04,2026-01-26T16:10:01
3,NK-22T6dTsc6umYgEA33vME3t,Organization,Digital Marketing Awards FZ LLC,SD Digital Marketing Awards;Swiss Dragons Mana...,NaN,ae,"Compass Building FDRK 2508, Al Shohada Road, A...",12119212;17002525,RUSSIA-EO14024 - Executive Order 14024 (Russia),NaN,NaN,US-RUSHAR,US OFAC Specially Designated Nationals (SDN) List,2024-08-23T16:10:02,2026-10-24T14:10:04,2026-01-26T16:10:01
4,NK-22cQD3a8GDRSuoW5mus3F3,Organization,EKVIK LIMITED LIABILITY COMPANY,NaN,NaN,ru,"Per. Dukhovskoi D. 17, Korpus 1, Floor 1 Podye...",1157746236966;7722320806,RUSSIA-EO14024 - Executive Order 14024 (Russia),NaN,NaN,US-RUSHAR,US OFAC Specially Designated Nationals (SDN) List,2024-10-30T14:10:02,2026-10-24T14:10:04,2026-01-26T16:10:01


In [ ]:
import random

# --- Частина 1: 12 довільних вигаданих імен звичайних клієнтів (для навчальної мети) ---
fictional_clients = pd.DataFrame({
    "full_name": [
        "Ivan Petrenko", "Maria Kovalenko", "John Smith",
        "Oleksandr Bondarenko", "Olena Tkachenko", "Andrii Melnyk",
        "Sarah Johnson", "Michael Brown", "Nataliya Shevchenko",
        "David Wilson", "Kateryna Moroz", "James Anderson",
    ],
    "country": [
        "Ukraine", "Ukraine", "USA",
        "Ukraine", "Ukraine", "Ukraine",
        "USA", "USA", "Ukraine",
        "USA", "Ukraine", "USA",
    ]
})

# --- Частина 2: 6 навмисно видозмінених реальних імен із sanctions_df ---
# Беремо 6 випадкових реальних записів із завантаженого санкційного переліку
# і застосовуємо просту видозміну написання (симуляція транслітераційної
# помилки чи неточного введення даних оператором)

name_column = "name" if "name" in sanctions_df.columns else "caption"
sampled_real_names = sanctions_df[name_column].dropna().sample(6, random_state=42).tolist()

def perturb_name(name):
    """Імітує типову помилку написання: переставляє слова, або прибирає одну літеру."""
    words = str(name).split()
    if len(words) >= 2 and random.random() < 0.5:
        # переставити перше і друге слово місцями (імітація "прізвище ім'я" замість "ім'я прізвище")
        words[0], words[1] = words[1], words[0]
        return " ".join(words)
    else:
        # прибрати одну випадкову літеру всередині імені
        idx = random.randint(1, len(name) - 2) if len(name) > 3 else 0
        return name[:idx] + name[idx + 1:]

perturbed_real_names = [perturb_name(n) for n in sampled_real_names]

perturbed_clients = pd.DataFrame({
    "full_name": perturbed_real_names,
    "country": ["Ukraine"] * 6,  # країну для цієї частини можна залишити довільною
})

# --- Об'єднання обох частин в один тестовий набір із 18 клієнтів ---
test_clients = pd.concat([fictional_clients, perturbed_clients], ignore_index=True)
test_clients

,full_name,country
0,Ivan Petrenko,Ukraine
1,Maria Kovalenko,Ukraine
2,John Smith,USA
3,Oleksandr Bondarenko,Ukraine
4,Olena Tkachenko,Ukraine
5,Andrii Melnyk,Ukraine
6,Sarah Johnson,USA
7,Michael Brown,USA
8,Nataliya Shevchenko,Ukraine
9,David Wilson,USA


In [ ]:
!pip install rapidfuzz --quiet
from rapidfuzz import process, fuzz

sanction_names = sanctions_df["name"].dropna().tolist()  # або "caption", залежно від структури файлу

def find_best_match(name):
    match = process.extractOne(name, sanction_names, scorer=fuzz.WRatio)
    return pd.Series({"matched_name": match[0], "similarity_score": match[1]})

test_clients[["matched_name", "similarity_score"]] = test_clients["full_name"].apply(find_best_match)
test_clients

,full_name,country,matched_name,similarity_score
0,Ivan Petrenko,Ukraine,Cristian Ivan ESPARRAGOZA GASTELUM,85.500000
1,Maria Kovalenko,Ukraine,Maria,90.000000
2,John Smith,USA,John Francis MORRISSEY,85.500000
3,Oleksandr Bondarenko,Ukraine,Oleksandr Sergeyevich KHODAKOVSKYY,85.500000
4,Olena Tkachenko,Ukraine,Olena Oleksandrivna Shapurova,85.500000
5,Andrii Melnyk,Ukraine,Andrii Leonidovych Derkach,85.500000
6,Sarah Johnson,USA,Sarah Jamal Muhammad al-Sayyid,85.500000
7,Michael Brown,USA,Michael IBARRA DIAZ JR.,85.500000
8,Nataliya Shevchenko,Ukraine,Yevgeny Aleksandrovich Shevchenko,85.500000
9,David Wilson,USA,David Duran Alvarez,85.500000


In [ ]:
high_risk_countries = ["North Korea", "Iran", "Syria"]  # приклад, визначити самостійно

test_clients["x1"] = (test_clients["similarity_score"] > 75).astype(int)
test_clients["x2"] = test_clients["similarity_score"]
test_clients["x3"] = test_clients["country"].isin(high_risk_countries).astype(int)

test_clients["RiskScore"] = (
    70 * test_clients["x1"] +
    0.2 * test_clients["x2"] +
    10 * test_clients["x3"]
)

test_clients.sort_values("RiskScore", ascending=False)

,full_name,country,matched_name,similarity_score,x1,x2,x3,RiskScore
14,Aktsionernoe Obshchestvo Nauchno Proizvdstvenn...,Ukraine,Aktsionernoe Obshchestvo Nauchno Proizvodstven...,99.224806,1,99.224806,0,89.844961
13,Oscar Enrique Bisca Gallego,Ukraine,Oscar Enrique Biosca Gallego,98.181818,1,98.181818,0,89.636364
16,Hosin SALIMI,Ukraine,Hosein SALIMI,96.000000,1,96.000000,0,89.200000
15,Petrochemical Urmia Company,Ukraine,Urmia Petrochemical Company,95.000000,1,95.000000,0,89.000000
17,Exim Maiwand Private Limited,Ukraine,Maiwand Exim Private Limited,95.000000,1,95.000000,0,89.000000
12,Khan Dilawar Nadir Khan,Ukraine,Dilawar Khan Nadir Khan,95.000000,1,95.000000,0,89.000000
1,Maria Kovalenko,Ukraine,Maria,90.000000,1,90.000000,0,88.000000
0,Ivan Petrenko,Ukraine,Cristian Ivan ESPARRAGOZA GASTELUM,85.500000,1,85.500000,0,87.100000
2,John Smith,USA,John Francis MORRISSEY,85.500000,1,85.500000,0,87.100000
3,Oleksandr Bondarenko,Ukraine,Oleksandr Sergeyevich KHODAKOVSKYY,85.500000,1,85.500000,0,87.100000


In [ ]:
thresholds = [60, 75, 90]
for t in thresholds:
    matches_found = (test_clients["similarity_score"] > t).sum()
    print(f"Поріг {t}: знайдено {matches_found} збігів")

Поріг 60: знайдено 18 збігів
Поріг 75: знайдено 17 збігів
Поріг 90: знайдено 6 збігів
